In [1]:
!git clone https://github.com/MahdhiS/hotel-cancellation.git

Cloning into 'hotel-cancellation'...
remote: Enumerating objects: 91, done.
remote: Counting objects: 100% (91/91), done.
remote: Compressing objects: 100% (69/69), done.
remote: Total 91 (delta 43), reused 49 (delta 17), pack-reused 0 (from 0)
Receiving objects: 100% (91/91), 10.21 MiB | 4.30 MiB/s, done.
Resolving deltas: 100% (43/43), done.


In [2]:
%cd /content/hotel-cancellation

/content/hotel-cancellation


In [3]:
!pwd
!ls

/content/hotel-cancellation
data  logs  models  notebooks  README.md  requirements.txt  src  template.ipynb


In [4]:
!ls data/processed/feature_engineered

hotel_test_fe.csv  hotel_train_fe.csv  hotel_val_fe.csv  init.txt


In [6]:
!mv /content/preprocessing.py /content/hotel-cancellation/src/preprocessing.py

mv: cannot stat '/content/preprocessing.py': No such file or directory


In [7]:
!ls src

data.py  evaluate.py  features.py  preprocessing.py


In [8]:
from src.preprocessing import (
    SharedPreprocessor,
    split_classifier_data
)

print("preprocessing.py imported successfully")

preprocessing.py imported successfully


#### P3-05
# Shared Preprocessing
#### M2 - Amri B.S.M
#### IT24102955

## 1. Setup

This notebook performs the shared, model-independent preprocessing used before model-specific pipelines.

The shared preprocessing covers:

- Preservation of a booking identifier for clustering and record tracking
- Removal of features that are not available at booking time
- Removal of invalid booking records
- Missing-value handling
- Rare-category grouping learned from training data only
- Creation of a consistent final feature set

Encoding, scaling, outlier treatment, transformations, and class-imbalance handling are not performed here. These are handled later inside model-specific pipelines.

In [9]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd()

print("Project root:", ROOT)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.preprocessing import (
    SharedPreprocessor,
    split_classifier_data
)

pd.set_option("display.max_columns", None)

FE_DIR = ROOT / "data" / "processed" / "feature_engineered"

print("Feature-engineered data directory:", FE_DIR)

Project root: /content/hotel-cancellation
Feature-engineered data directory: /content/hotel-cancellation/data/processed/feature_engineered


## 2. Load Feature-Engineered Data

The train, validation, and test datasets created during the feature-engineering stage are loaded here.

All preprocessing rules that require learning from data will be fitted using the training set only.

In [10]:
train_df = pd.read_csv(FE_DIR / "hotel_train_fe.csv")
val_df = pd.read_csv(FE_DIR / "hotel_val_fe.csv")
test_df = pd.read_csv(FE_DIR / "hotel_test_fe.csv")

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

Train shape: (58990, 42)
Validation shape: (10544, 42)
Test shape: (17406, 42)


In [11]:
train_df.head()

,Unnamed: 0.1,Unnamed: 0,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,arrival_date_month_num,arrival_date,booking_date,total_nights,total_guests,has_kids,season,arrival_day_of_week
0,0,0,Resort Hotel,0,342,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,3,No Deposit,0.0,0.0,0,Transient,0.0,0,0,Check-Out,2015-07-01,7,2015-07-01,2014-07-24,0,2.0,0,Summer,Wednesday
1,1,1,Resort Hotel,0,737,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,4,No Deposit,0.0,0.0,0,Transient,0.0,0,0,Check-Out,2015-07-01,7,2015-07-01,2013-06-24,0,2.0,0,Summer,Wednesday
2,2,2,Resort Hotel,0,7,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Direct,Direct,0,0,0,A,C,0,No Deposit,0.0,0.0,0,Transient,75.0,0,0,Check-Out,2015-07-02,7,2015-07-01,2015-06-24,1,1.0,0,Summer,Wednesday
3,3,3,Resort Hotel,0,13,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Corporate,Corporate,0,0,0,A,A,0,No Deposit,304.0,0.0,0,Transient,75.0,0,0,Check-Out,2015-07-02,7,2015-07-01,2015-06-18,1,1.0,0,Summer,Wednesday
4,4,4,Resort Hotel,0,14,2015,July,27,1,0,2,2,0.0,0,BB,GBR,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,240.0,0.0,0,Transient,98.0,0,1,Check-Out,2015-07-03,7,2015-07-01,2015-06-17,2,2.0,0,Summer,Wednesday


In [12]:
print(train_df.columns.tolist())

['Unnamed: 0.1', 'Unnamed: 0', 'hotel', 'is_canceled', 'lead_time', 'arrival_date_year', 'arrival_date_month', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'meal', 'country', 'market_segment', 'distribution_channel', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'reserved_room_type', 'assigned_room_type', 'booking_changes', 'deposit_type', 'agent', 'company', 'days_in_waiting_list', 'customer_type', 'adr', 'required_car_parking_spaces', 'total_of_special_requests', 'reservation_status', 'reservation_status_date', 'arrival_date_month_num', 'arrival_date', 'booking_date', 'total_nights', 'total_guests', 'has_kids', 'season', 'arrival_day_of_week']


In [13]:
unnamed_cols = [
    col for col in train_df.columns
    if col.startswith("Unnamed:")
]

print("Unnamed columns found:", unnamed_cols)

for col in unnamed_cols:
    print(f"\nColumn: {col}")
    print("Unique:", train_df[col].is_unique)
    print("Number of unique values:", train_df[col].nunique())
    print("First 10 values:")
    print(train_df[col].head(10).tolist())

Unnamed columns found: ['Unnamed: 0.1', 'Unnamed: 0']

Column: Unnamed: 0.1
Unique: True
Number of unique values: 58990
First 10 values:
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

Column: Unnamed: 0
Unique: True
Number of unique values: 58990
First 10 values:
[0, 1, 2, 3, 4, 6, 7, 8, 9, 10]


In [14]:
common_unnamed = [
    col for col in train_df.columns
    if col.startswith("Unnamed:")
    and col in val_df.columns
    and col in test_df.columns
]

for col in common_unnamed:

    combined = pd.concat([
        train_df[[col]],
        val_df[[col]],
        test_df[[col]]
    ], ignore_index=True)

    print(f"\n{col}")
    print("Total rows:", len(combined))
    print("Unique values:", combined[col].nunique())
    print("Globally unique:", combined[col].is_unique)


Unnamed: 0.1
Total rows: 86940
Unique values: 86940
Globally unique: True

Unnamed: 0
Total rows: 86940
Unique values: 86940
Globally unique: True


## 3. Booking Identifier Handling

Two automatically generated CSV index columns were identified:

- `Unnamed: 0.1`
- `Unnamed: 0`

Both are globally unique across the train, validation, and test datasets.

`Unnamed: 0` is retained as `booking_id` because its non-continuous values indicate that it preserves the earlier record identity after data-cleaning operations. `Unnamed: 0.1` appears to be a later CSV export index and is therefore removed.

The `booking_id` field is retained for record tracking and clustering purposes, but it will not be used as a supervised classification predictor.

## 4. Apply Shared Preprocessing

The `SharedPreprocessor` is fitted only on the training dataset.

During fitting, it:

- Preserves the booking identifier
- Removes leakage-risk and helper columns
- Removes invalid booking records
- Learns missing-value replacement rules from training data
- Learns frequent categories from training data

The learned rules are then applied unchanged to the validation and test datasets.

In [15]:
prep = SharedPreprocessor(
    rare_threshold=0.01,
    rare_min_unique=15
)

print(prep)

SharedPreprocessor(rare_threshold=0.01, rare_min_unique=15, fill_values_={}, frequent_categories_={}, output_columns_=[], output_columns_with_country_=[], fitted_=False)


In [16]:
train_prepared = prep.fit_transform(train_df)

val_prepared = prep.transform(val_df)

test_prepared = prep.transform(test_df)

In [17]:
shape_summary = pd.DataFrame({
    "Dataset": ["Train", "Validation", "Test"],
    "Before Rows": [
        len(train_df),
        len(val_df),
        len(test_df)
    ],
    "After Rows": [
        len(train_prepared),
        len(val_prepared),
        len(test_prepared)
    ],
    "Before Columns": [
        train_df.shape[1],
        val_df.shape[1],
        test_df.shape[1]
    ],
    "After Columns": [
        train_prepared.shape[1],
        val_prepared.shape[1],
        test_prepared.shape[1]
    ]
})

shape_summary["Rows Removed"] = (
    shape_summary["Before Rows"]
    - shape_summary["After Rows"]
)

shape_summary

,Dataset,Before Rows,After Rows,Before Columns,After Columns,Rows Removed
0,Train,58990,58192,42,29,798
1,Validation,10544,10458,42,29,86
2,Test,17406,17324,42,29,82


In [18]:
print(
    "Same columns across all splits:",
    list(train_prepared.columns)
    == list(val_prepared.columns)
    == list(test_prepared.columns)
)

print(
    "booking_id present:",
    "booking_id" in train_prepared.columns
)

print(
    "Target present:",
    "is_canceled" in train_prepared.columns
)

print(
    "Country present in standard classifier dataset:",
    "country" in train_prepared.columns
)

print(
    "Remaining missing values in train:",
    train_prepared.isna().sum().sum()
)

print(
    "Remaining missing values in validation:",
    val_prepared.isna().sum().sum()
)

print(
    "Remaining missing values in test:",
    test_prepared.isna().sum().sum()
)

Same columns across all splits: True
booking_id present: True
Target present: True
Country present in standard classifier dataset: False
Remaining missing values in train: 0
Remaining missing values in validation: 0
Remaining missing values in test: 0


In [19]:
leakage_columns = [
    "assigned_room_type",
    "booking_changes",
    "days_in_waiting_list",
    "customer_type",
    "required_car_parking_spaces",
    "total_of_special_requests",
    "reservation_status",
    "reservation_status_date"
]

for col in leakage_columns:
    print(
        f"{col:<35}",
        "REMOVED" if col not in train_prepared.columns else "STILL PRESENT"
    )

assigned_room_type                  REMOVED
booking_changes                     REMOVED
days_in_waiting_list                REMOVED
customer_type                       REMOVED
required_car_parking_spaces         REMOVED
total_of_special_requests           REMOVED
reservation_status                  REMOVED
reservation_status_date             REMOVED


In [20]:
helper_columns = [
    "Unnamed: 0.1",
    "Unnamed: 0",
    "arrival_date_month_num",
    "arrival_date",
    "booking_date"
]

for col in helper_columns:
    print(
        f"{col:<30}",
        "REMOVED" if col not in train_prepared.columns else "STILL PRESENT"
    )

Unnamed: 0.1                   REMOVED
Unnamed: 0                     REMOVED
arrival_date_month_num         REMOVED
arrival_date                   REMOVED
booking_date                   REMOVED


In [21]:
print("Train booking_id unique:",
      train_prepared["booking_id"].is_unique)

print("Validation booking_id unique:",
      val_prepared["booking_id"].is_unique)

print("Test booking_id unique:",
      test_prepared["booking_id"].is_unique)

all_ids = pd.concat([
    train_prepared["booking_id"],
    val_prepared["booking_id"],
    test_prepared["booking_id"]
], ignore_index=True)

print("booking_id globally unique:",
      all_ids.is_unique)

Train booking_id unique: True
Validation booking_id unique: True
Test booking_id unique: True
booking_id globally unique: True


In [22]:
print("Final number of columns:", len(train_prepared.columns))

for i, col in enumerate(train_prepared.columns, start=1):
    print(f"{i:02d}. {col}")

Final number of columns: 29
01. booking_id
02. hotel
03. is_canceled
04. lead_time
05. arrival_date_year
06. arrival_date_month
07. arrival_date_week_number
08. arrival_date_day_of_month
09. stays_in_weekend_nights
10. stays_in_week_nights
11. adults
12. children
13. babies
14. meal
15. market_segment
16. distribution_channel
17. is_repeated_guest
18. previous_cancellations
19. previous_bookings_not_canceled
20. reserved_room_type
21. deposit_type
22. agent
23. company
24. adr
25. total_nights
26. total_guests
27. has_kids
28. season
29. arrival_day_of_week


## 5. Validate Invalid Booking Removal

The shared preprocessing removes records that are not suitable as normal hotel bookings for modelling.

The current rules remove:

- bookings with no adult guest
- bookings with zero total nights

Bookings with no adults are investigated further to distinguish completely empty bookings from children-only bookings.

The following checks verify that the observed row reduction is caused only by these predefined rules.

In [23]:
def invalid_booking_summary(df):
    no_adult = df["adults"] <= 0
    zero_night = df["total_nights"] <= 0

    zero_guests = df["total_guests"] <= 0

    children_only = (
        (df["adults"] <= 0)
        & ((df["children"].fillna(0) + df["babies"].fillna(0)) > 0)
    )

    removed_by_rule = no_adult | zero_night

    return {
        "Total rows": len(df),
        "No adult": no_adult.sum(),
        "Zero guests": zero_guests.sum(),
        "Children-only": children_only.sum(),
        "Zero nights": zero_night.sum(),
        "No adult AND zero nights": (no_adult & zero_night).sum(),
        "Rows removed by union": removed_by_rule.sum()
    }


invalid_summary = pd.DataFrame({
    "Train": invalid_booking_summary(train_df),
    "Validation": invalid_booking_summary(val_df),
    "Test": invalid_booking_summary(test_df)
}).T

invalid_summary

,Total rows,No adult,Zero guests,Children-only,Zero nights,No adult AND zero nights,Rows removed by union
Train,58990,282,112,170,550,34,798
Validation,10544,52,30,22,50,16,86
Test,17406,46,19,27,42,6,82


In [24]:
expected_removed = {
    "Train": len(train_df) - len(train_prepared),
    "Validation": len(val_df) - len(val_prepared),
    "Test": len(test_df) - len(test_prepared)
}

rule_removed = {
    "Train": (
        (train_df["adults"] <= 0)
        | (train_df["total_nights"] <= 0)
    ).sum(),

    "Validation": (
        (val_df["adults"] <= 0)
        | (val_df["total_nights"] <= 0)
    ).sum(),

    "Test": (
        (test_df["adults"] <= 0)
        | (test_df["total_nights"] <= 0)
    ).sum()
}

for split in expected_removed:
    print(
        f"{split:<12}",
        "Observed removed:", expected_removed[split],
        "| Rule-based:", rule_removed[split],
        "| Match:", expected_removed[split] == rule_removed[split]
    )

Train        Observed removed: 798 | Rule-based: 798 | Match: True
Validation   Observed removed: 86 | Rule-based: 86 | Match: True
Test         Observed removed: 82 | Rule-based: 82 | Match: True


In [25]:
for name, df in {
    "Train": train_prepared,
    "Validation": val_prepared,
    "Test": test_prepared
}.items():

    print(f"\n{name}")

    print(
        "Rows with adults <= 0:",
        (df["adults"] <= 0).sum()
    )

    print(
        "Rows with total_nights <= 0:",
        (df["total_nights"] <= 0).sum()
    )


Train
Rows with adults <= 0: 0
Rows with total_nights <= 0: 0

Validation
Rows with adults <= 0: 0
Rows with total_nights <= 0: 0

Test
Rows with adults <= 0: 0
Rows with total_nights <= 0: 0


In [26]:
print("Columns with learned rare-category rules:")

for col, categories in prep.frequent_categories_.items():
    print(
        f"{col:<25}",
        "frequent categories retained:",
        len(categories)
    )

Columns with learned rare-category rules:
country                   frequent categories retained: 12
agent                     frequent categories retained: 11
company                   frequent categories retained: 2


In [27]:
for col, categories in prep.frequent_categories_.items():
    print(f"\n{col}")
    print(sorted(categories))


country
['BEL', 'BRA', 'CHE', 'DEU', 'ESP', 'FRA', 'GBR', 'IRL', 'ITA', 'NLD', 'PRT', 'USA']

agent
['0.0', '1.0', '14.0', '240.0', '241.0', '250.0', '28.0', '6.0', '7.0', '8.0', '9.0']

company
['0.0', '40.0']


In [28]:
for col in prep.frequent_categories_:

    print(f"\n{col}")

    print(
        "Train Other:",
        (train_prepared[col] == "Other").sum()
        if col in train_prepared.columns
        else "Column excluded from standard dataset"
    )

    print(
        "Validation Other:",
        (val_prepared[col] == "Other").sum()
        if col in val_prepared.columns
        else "Column excluded from standard dataset"
    )

    print(
        "Test Other:",
        (test_prepared[col] == "Other").sum()
        if col in test_prepared.columns
        else "Column excluded from standard dataset"
    )


country
Train Other: Column excluded from standard dataset
Validation Other: Column excluded from standard dataset
Test Other: Column excluded from standard dataset

agent
Train Other: 11887
Validation Other: 1715
Test Other: 3283

company
Train Other: 2914
Validation Other: 701
Test Other: 551


## 6. Shared Preprocessing Summary

The validation checks confirmed that shared preprocessing behaves consistently across all three data splits.

Invalid-record removal was fully accounted for by the predefined rules:

- bookings with no adult guest
- bookings with zero total nights

No additional unexplained rows were removed.

Rare-category grouping was learned exclusively from the training data using a 1% frequency threshold for high-cardinality categorical attributes. Categories below the threshold, including unseen categories in validation and test data, are grouped as `Other`.

The resulting train, validation, and test datasets use the same 29-column schema and contain no missing values.

In [29]:
OUT_DIR = ROOT / "data" / "processed" / "shared_prepared"
OUT_DIR.mkdir(parents=True, exist_ok=True)

train_path = OUT_DIR / "hotel_train_prepared.csv"
val_path = OUT_DIR / "hotel_val_prepared.csv"
test_path = OUT_DIR / "hotel_test_prepared.csv"

train_prepared.to_csv(train_path, index=False)
val_prepared.to_csv(val_path, index=False)
test_prepared.to_csv(test_path, index=False)

print("Saved:")
print(train_path)
print(val_path)
print(test_path)

Saved:
/content/hotel-cancellation/data/processed/shared_prepared/hotel_train_prepared.csv
/content/hotel-cancellation/data/processed/shared_prepared/hotel_val_prepared.csv
/content/hotel-cancellation/data/processed/shared_prepared/hotel_test_prepared.csv


In [30]:
saved_train = pd.read_csv(train_path)
saved_val = pd.read_csv(val_path)
saved_test = pd.read_csv(test_path)

print("Train:", saved_train.shape)
print("Validation:", saved_val.shape)
print("Test:", saved_test.shape)

print(
    "\nSchemas identical:",
    list(saved_train.columns)
    == list(saved_val.columns)
    == list(saved_test.columns)
)

print(
    "Missing values:",
    saved_train.isna().sum().sum(),
    saved_val.isna().sum().sum(),
    saved_test.isna().sum().sum()
)

Train: (58192, 29)
Validation: (10458, 29)
Test: (17324, 29)

Schemas identical: True
Missing values: 0 0 0


In [31]:
!sed -n '1,260p' src/evaluate.py

import numpy as np
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)


def evaluate(model, X_val, y_val, threshold=None):
    y_true = np.asarray(y_val)

    if hasattr(model, "predict_proba"):
        score = model.predict_proba(X_val)[:, 1]
        default_threshold = 0.5
    elif hasattr(model, "decision_function"):
        score = model.decision_function(X_val)
        default_threshold = 0.0
    else:
        raise ValueError("model needs predict_proba or decision_function")

    thr = default_threshold if threshold is None else threshold
    pred = (score >= thr).astype(int)

    return {
        "roc_auc": roc_auc_score(y_true, score),
        "pr_auc": average_precision_score(y_true, score),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "f1": f1_score(y_true, pred, zero_division=0),

## 7. End-to-End Logistic Regression Smoke Test

A simple Logistic Regression model is used to verify that the shared-prepared training data can be consumed successfully by a classification pipeline.

This is only an end-to-end validation of the shared preprocessing stage. Model-specific preprocessing such as encoding and scaling is kept inside the temporary Logistic Regression pipeline and is not added to the shared preprocessing module.

Cross-validation is performed using the training set only. The test set remains untouched.

In [32]:
X_train, y_train, train_booking_ids = split_classifier_data(train_prepared)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("booking_id in predictors:", "booking_id" in X_train.columns)
print("is_canceled in predictors:", "is_canceled" in X_train.columns)
print("country in predictors:", "country" in X_train.columns)

X_train shape: (58192, 27)
y_train shape: (58192,)
booking_id in predictors: False
is_canceled in predictors: False
country in predictors: False


In [33]:
numeric_features = X_train.select_dtypes(
    include=["number", "bool"]
).columns.tolist()

categorical_features = [
    col for col in X_train.columns
    if col not in numeric_features
]

print("Numeric features:", len(numeric_features))
print(numeric_features)

print("\nCategorical features:", len(categorical_features))
print(categorical_features)

Numeric features: 16
['lead_time', 'arrival_date_year', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'adr', 'total_nights', 'total_guests', 'has_kids']

Categorical features: 11
['hotel', 'arrival_date_month', 'meal', 'market_segment', 'distribution_channel', 'reserved_room_type', 'deposit_type', 'agent', 'company', 'season', 'arrival_day_of_week']


In [36]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate

In [37]:
numeric_transformer = Pipeline([
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore"
        )
    )
])

model_preprocessor = ColumnTransformer([
    (
        "numeric",
        numeric_transformer,
        numeric_features
    ),
    (
        "categorical",
        categorical_transformer,
        categorical_features
    )
])

logistic_smoke_test = Pipeline([
    ("preprocessor", model_preprocessor),
    (
        "model",
        LogisticRegression(
            max_iter=2000,
            random_state=42
        )
    )
])

In [38]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_results = cross_validate(
    logistic_smoke_test,
    X_train,
    y_train,
    cv=cv,
    scoring={
        "roc_auc": "roc_auc",
        "pr_auc": "average_precision"
    },
    n_jobs=-1,
    return_train_score=False
)

In [39]:
cv_summary = pd.DataFrame({
    "Fold": range(1, 6),
    "ROC-AUC": cv_results["test_roc_auc"],
    "PR-AUC": cv_results["test_pr_auc"]
})

cv_summary

,Fold,ROC-AUC,PR-AUC
0,1,0.745826,0.530432
1,2,0.749822,0.538142
2,3,0.749523,0.534189
3,4,0.754825,0.545508
4,5,0.754135,0.537944


In [40]:
print(
    "Mean ROC-AUC:",
    round(cv_results["test_roc_auc"].mean(), 4)
)

print(
    "ROC-AUC std:",
    round(cv_results["test_roc_auc"].std(), 4)
)

print(
    "Mean PR-AUC:",
    round(cv_results["test_pr_auc"].mean(), 4)
)

print(
    "PR-AUC std:",
    round(cv_results["test_pr_auc"].std(), 4)
)

Mean ROC-AUC: 0.7508
ROC-AUC std: 0.0033
Mean PR-AUC: 0.5372
PR-AUC std: 0.005


## 8. End-to-End Validation Result

The shared-prepared training data was successfully tested using a simple Logistic Regression pipeline with 5-fold stratified cross-validation.

Model-specific preprocessing was kept inside the temporary model pipeline:

- numerical features were standardized
- categorical features were one-hot encoded
- Logistic Regression was used only as a smoke test

The shared preprocessing itself therefore remains independent of any specific classification algorithm.

### Cross-Validation Results

- Mean ROC-AUC: **0.7508**
- ROC-AUC standard deviation: **0.0033**
- Mean PR-AUC: **0.5372**
- PR-AUC standard deviation: **0.0050**

The small variation between folds indicates that the prepared training dataset can be processed consistently by a classification pipeline.

These results are not treated as the final Logistic Regression model performance. Full model development, tuning, imbalance handling, and evaluation are performed separately during the modelling phase.

In [ ]:
assert list(train_prepared.columns) == list(val_prepared.columns)
assert list(train_prepared.columns) == list(test_prepared.columns)

assert train_prepared.isna().sum().sum() == 0
assert val_prepared.isna().sum().sum() == 0
assert test_prepared.isna().sum().sum() == 0

assert train_prepared["booking_id"].is_unique
assert val_prepared["booking_id"].is_unique
assert test_prepared["booking_id"].is_unique

assert "reservation_status" not in train_prepared.columns
assert "reservation_status_date" not in train_prepared.columns
assert "assigned_room_type" not in train_prepared.columns

assert "booking_id" not in X_train.columns
assert "is_canceled" not in X_train.columns

print("All shared preprocessing validation checks passed.")

## 9. Conclusion

The shared preprocessing stage was completed successfully.

The final preprocessing workflow:

1. preserves a unique booking identifier for clustering and record tracking
2. removes fields that are not permitted for booking-time cancellation prediction
3. removes invalid zero-adult and zero-night booking records
4. applies missing-value rules learned from training data only
5. groups rare high-cardinality categories using training-data frequencies only
6. produces an identical 29-column schema across train, validation, and test data
7. excludes `booking_id` and the target from supervised model predictors
8. excludes `country` from the standard classifier dataset while allowing it to be retained separately for the later responsible-AI comparison
9. successfully passes an end-to-end Logistic Regression cross-validation smoke test

Encoding, scaling, outlier treatment, skew transformations, class-imbalance handling, model tuning, and threshold selection remain outside shared preprocessing and will be handled during the appropriate later modelling stages.

The shared preprocessing outputs are ready for downstream classification and clustering workflows.